# Parking-capacity — entraînement YOLOv8-seg satellite (APKLOT)

Ajustez **`ZIP_PATH`**, **`OUTPUT_RUN`** (sur Drive pour ne pas perdre les checkpoints), **`SAVE_PERIOD_EPOCHS`** (checkpoint tous les N epochs ; 0 = désactivé).

## A. Monter Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Chemin vers parking_capacity_colab.zip sur votre Drive
ZIP_PATH = "/content/drive/MyDrive/parking_capacity_colab.zip"
ROOT = "/content/parking_colab"
# Sortie entraînement sur Drive : checkpoints Ultralytics y sont écrits directement
OUTPUT_RUN = "/content/drive/MyDrive/colab_runs/yolo_seg"
SAVE_PERIOD_EPOCHS = 5  # 0 pour désactiver ; sinon checkpoint tous les N epochs dans OUTPUT_RUN/yolo_train/weights/
EPOCHS = 50
MODEL = "yolov8m-seg.pt"
IMGSZ = 640
BENCHMARK_MOSAIC = "/content/drive/MyDrive/colab_benchmark_mosaic.png"

## B. Décompresser `parking_capacity_colab.zip`

In [ ]:
import os
import shutil
import subprocess

shutil.rmtree(ROOT, ignore_errors=True)
os.makedirs(os.path.dirname(ROOT), exist_ok=True)
subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", ROOT], check=True)
os.chdir(os.path.join(ROOT, "project_snapshot"))
print("cwd:", os.getcwd())

## C. Installation propre du package
Désinstalle l’ancienne version pip puis réinstalle en mode éditable avec les extras vision / satellite.

In [ ]:
!pip uninstall -y parking-capacity
!pip install -q -r ../requirements_colab.txt
!pip install -q -e ".[train_satellite,vision]"

## D. Environnement Colab (Python, CUDA, torch, Ultralytics, GPU)

In [ ]:
import subprocess
import sys

print("Python:", sys.version)

try:
    import torch

    print("torch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA version (torch):", torch.version.cuda)
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch import:", e)

try:
    import ultralytics

    print("ultralytics:", ultralytics.__version__)
except Exception as e:
    print("ultralytics:", e)

subprocess.run(["nvidia-smi"], check=False)

## E. Validation post-install (synchronisation notebook ↔ package)
Vérifie la présence des options CLI attendues dans les textes d’aide.

In [ ]:
import subprocess


def _help(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return (r.stdout or "") + (r.stderr or "")


helps = {
    "parking-capacity --help": _help(["parking-capacity", "--help"]),
    "parking-capacity datasets-prepare --help": _help(
        ["parking-capacity", "datasets-prepare", "--help"]
    ),
    "parking-capacity inspect-dataset --help": _help(
        ["parking-capacity", "inspect-dataset", "--help"]
    ),
}

missing = []
if "benchmark-dataset-mosaics" not in helps["parking-capacity --help"]:
    missing.append(("parking-capacity --help", "benchmark-dataset-mosaics"))
if "--apklot-view" not in helps["parking-capacity datasets-prepare --help"]:
    missing.append(("datasets-prepare --help", "--apklot-view"))

train_h = _help(["parking-capacity", "train-yolo-seg", "--help"])
if "--force-incompatible-dataset" not in train_h:
    missing.append(("train-yolo-seg --help", "--force-incompatible-dataset"))

if missing:
    for ctx, token in missing:
        print(f"Manquant dans {ctx}: {token}")
    raise RuntimeError(
        "Le notebook et le package exporté ne sont pas synchronisés. "
        "Refaites un export local : parking-capacity export-colab-training … puis ré-uploadez le ZIP."
    )
print("Validation CLI : OK")

## F. Statistiques du jeu préparé

In [ ]:
!parking-capacity dataset-stats --dataset apklot

## G. Téléchargement / préparation APKLOT (vue satellite)
Si le ZIP ne contenait pas les données (`DATASET_TOO_LARGE.txt`) ou si les dossiers manquent, exécutez ces lignes (peut être long).

In [ ]:
# !parking-capacity datasets-download --dataset apklot
!parking-capacity datasets-prepare --dataset apklot --apklot-view satellite

## H. Inspection APKLOT (`dataset_type`, satellite, garde-fous)
Après préparation, `dataset_prepare_meta.json` indique si l’entraînement « vue satellite » est pertinent.

In [ ]:
import json
import subprocess

r = subprocess.run(
    ["parking-capacity", "inspect-dataset", "--dataset", "apklot"],
    check=True,
    capture_output=True,
    text=True,
)
info = json.loads(r.stdout)
print(json.dumps(info, indent=2, ensure_ascii=False))

dt = info.get("dataset_type")
pm = info.get("prepare_meta") or {}
sat_ok = pm.get("satellite_segmentation_suitable")
by_view = pm.get("images_by_view") or {}
n_sat = int(by_view.get("satellite", 0))

print("dataset_type:", dt)
print("satellite_segmentation_suitable:", sat_ok)
print("images_by_view:", by_view)
print("images satellite (préparé):", n_sat)

ALLOW_SAT_TRAIN = bool(sat_ok) if sat_ok is not None else True
if sat_ok is False:
    print(
        "\n>>> Diagnostic : APKLOT préparé sans image satellite exploitable pour la segmentation orthophoto. "
        "Vérifiez raw/apklot (« 1. Satellite », git lfs pull) ou utilisez --apklot-view all en connaissance de cause, "
        "ou un jeu DOTA/xView/SpaceNet.\n"
    )

## I. Mosaïque benchmark (jeux satellite)

In [ ]:
import os
import subprocess

os.makedirs(os.path.dirname(BENCHMARK_MOSAIC), exist_ok=True)
subprocess.run(
    [
        "parking-capacity",
        "benchmark-dataset-mosaics",
        "--out",
        BENCHMARK_MOSAIC,
        "--datasets",
        "apklot,dota,xview,spacenet",
        "--samples",
        "4",
    ],
    check=True,
)
print("Mosaïque écrite :", BENCHMARK_MOSAIC)

## J. Entraînement YOLOv8-seg
Bloqué si `satellite_segmentation_suitable` est faux (section H). Pour ignorer : ajoutez `--force-incompatible-dataset` dans la commande ci-dessous.

In [ ]:
import os
import subprocess

if not ALLOW_SAT_TRAIN:
    raise RuntimeError(
        "Entraînement satellite bloqué : satellite_segmentation_suitable=false. "
        "Corrigez les données APKLOT ou utilisez explicitement --force-incompatible-dataset si vous assumez le risque."
    )

cmd = [
    "parking-capacity",
    "train-yolo-seg",
    "--dataset",
    "apklot",
    "--model",
    MODEL,
    "--epochs",
    str(EPOCHS),
    "--imgsz",
    str(IMGSZ),
    "--output-dir",
    OUTPUT_RUN,
    "--save-period",
    str(SAVE_PERIOD_EPOCHS),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("Checkpoints :", os.path.join(OUTPUT_RUN, "yolo_train", "weights"))

## K. Reprise après interruption (`--resume`)

In [ ]:
import os
import subprocess

LAST_PT = os.path.join(OUTPUT_RUN, "yolo_train", "weights", "last.pt")
# Décommenter pour reprendre :
# subprocess.run(
#     [
#         "parking-capacity",
#         "train-yolo-seg",
#         "--dataset",
#         "apklot",
#         "--resume",
#         "--weights",
#         LAST_PT,
#         "--epochs",
#         str(EPOCHS),
#         "--imgsz",
#         str(IMGSZ),
#         "--output-dir",
#         OUTPUT_RUN,
#         "--save-period",
#         str(SAVE_PERIOD_EPOCHS),
#     ],
#     check=True,
# )
print("last.pt attendu:", LAST_PT)

## L. Évaluation val / test (Ultralytics)

In [ ]:
import glob

weights_glob = OUTPUT_RUN + "/**/weights/best.pt"
best_list = sorted(glob.glob(weights_glob, recursive=True))
assert best_list, "Aucun best.pt trouvé sous OUTPUT_RUN"
BEST_PT = best_list[-1]

yaml_candidates = [
    "../datasets/prepared/apklot/yolo_seg_dataset/dataset.yaml",
    "data/datasets/prepared/apklot/yolo_seg_dataset/dataset.yaml",
]
import os as _os

DATA_YAML = next((p for p in yaml_candidates if _os.path.isfile(p)), yaml_candidates[0])
print("BEST_PT", BEST_PT)
print("DATA_YAML", DATA_YAML)

In [ ]:
import subprocess

subprocess.run(
    ["yolo", "segment", "val", f"model={BEST_PT}", f"data={DATA_YAML}", "split=val"],
    check=True,
)
subprocess.run(
    ["yolo", "segment", "val", f"model={BEST_PT}", f"data={DATA_YAML}", "split=test"],
    check=True,
)

## M. Exporter résultats vers Drive (copie explicite)

In [ ]:
import os
import shutil
from pathlib import Path

EXPORT_DIR = "/content/drive/MyDrive/colab_yolo_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

bp = Path(BEST_PT).resolve()
parts = bp.parts
if "yolo_train" in parts:
    i = parts.index("yolo_train")
    run_root = Path(*parts[:i])
else:
    run_root = bp.parent.parent

yt = run_root / "yolo_train"
scan_dirs = [yt, run_root]
for base in scan_dirs:
    for rel in ("results.csv", "args.yaml", "train_metrics.json"):
        src = base / rel
        if src.is_file():
            shutil.copy2(src, os.path.join(EXPORT_DIR, rel))

weights_dir = bp.parent
for w in ("best.pt", "last.pt"):
    p = weights_dir / w
    if p.is_file():
        shutil.copy2(p, os.path.join(EXPORT_DIR, w))

tm = run_root / "train_metrics.json"
if tm.is_file():
    shutil.copy2(tm, os.path.join(EXPORT_DIR, "train_metrics_run.json"))

for name in ("sample_overlays", "sample_masks"):
    sd = run_root / name
    if sd.is_dir():
        shutil.copytree(sd, os.path.join(EXPORT_DIR, name), dirs_exist_ok=True)

print("Export →", EXPORT_DIR)

## N. Test orthophoto réelle

In [ ]:
import os
import subprocess

TEST_OUT = "/content/drive/MyDrive/colab_test_seg"
os.makedirs(TEST_OUT, exist_ok=True)
subprocess.run(
    [
        "parking-capacity",
        "test-segmentation-real",
        "--address",
        "2 Bd Industriel, 76270 Neufchâtel-en-Bray",
        "--weights",
        BEST_PT,
        "--out",
        TEST_OUT,
    ],
    check=True,
)

## O. Cohérence build (optionnel)
Compare la version pip au fichier `build_info.json` extrait dans le ZIP.

In [ ]:
!parking-capacity doctor-build --export-dir /content/parking_colab